# Pontificia Universidad Javeriana

<img src="assets/javeriana.png" alt="drawing" width="200"/>

<h1>Procesamiento de Datos a Gran Escala</h1>

### Fecha: 9 de septiembre de 2026

### Autor: Lorenzo Ramirez Calderon (ID: 20550074)

### Docente: John Jairo Corredor

# Apache Spark - Beginner Tutorial

Firstly, Thank you for checking this tutorial notebook. By now you must have read a lot about Apache Spark and its ML Library.So, I won't bore you with the introduction to Apache-Spark or even the library details. We shall move straight to the interesting stuff i.e. coding.

In this tuorial notebook we will try to compare some of the Apache Spark's Classification Algorithms in an easy way to make predictions. And since this is a beginner's tutorial, we will use the Iris Flower Dataset aka the beginner's dataset in machine learning.

**A quick summary:**

* Import Libraries
* Build Spark Session
* Data Load
* Data Exploration & Preparation
* Feature Engineering
* Data Scaling
* Data Split
* Build, Train & Evaluate Model


In [4]:
#install Apache Spark
!pip install pyspark --quiet

<div style="
    border-left: 5px solid #2E7D32;
    padding: 14px 18px;
    margin: 20px 0;
    border-radius: 6px;
">
<p style="
    font-size: 1.30em;
    font-weight: 700;
    margin: 0 0 8px 0;
">
<strong>Nota del autor</strong>

<p>
Instalacion de Libreria PySpark. PySpark permite utilizar la API de Python para trabajar con el motor distribuido de Apache Spark, haciendo posible procesar grandes volúmenes de datos mediante ejecución paralela con el servidor que previamente configuramos.
</p>

</div>

In [5]:
#install Numpy, Pandas, Scikit-learn, and Tabulate
!pip install numpy --quiet
!pip install pandas --quiet
!pip install scikit-learn --quiet
!pip install tabulate --quiet

## Importing Libraries

In [6]:
#Generic Libraries
import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

#Apache Spark Libraries
import pyspark
from pyspark.sql import SparkSession

#Apache Spark ML CLassifier Libraries
from pyspark.ml.classification import DecisionTreeClassifier,RandomForestClassifier,NaiveBayes

#Apache Spark Evaluation Library
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

#Apache Spark Features libraries
from pyspark.ml.feature import StandardScaler,StringIndexer, VectorAssembler, VectorIndexer, OneHotEncoder

#Apache Spark Pipelin Library
from pyspark.ml import Pipeline

# Apache Spark `DenseVector`
from pyspark.ml.linalg import DenseVector

#Data Split Libraries
import sklearn
from sklearn.model_selection import train_test_split


#Tabulating Data
from tabulate import tabulate

#Garbage
import gc

## Build Spark Session

In [7]:
#Building Spark Session
spark = (SparkSession.builder
                  .appName('Apache Spark Beginner Tutorial')
                  .config("spark.executor.memory", "1G")
                  .config("spark.executor.cores","4")
                  .getOrCreate())

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/09/20 15:51:01 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


<div style="
    border-left: 5px solid #2E7D32;
    padding: 14px 18px;
    margin: 20px 0;
    border-radius: 6px;
">
<p style="
    font-size: 1.30em;
    font-weight: 700;
    margin: 0 0 8px 0;
">
<strong>Nota del autor</strong>

<p>
<code>SparkSession</code> es el punto de entrada principal para trabajar con Spark desde Python. A través de ella se configura la aplicación y se habilita el acceso a funcionalidades. En este caso también se establecen recursos para la ejecución, como la memoria y los núcleos.

</p>

</div>

In [8]:
spark.sparkContext.setLogLevel('INFO')

In [9]:
spark.version

'4.0.4'

## Data Load

In [10]:
url = 'iris.csv'

data = spark.read.format("csv") \
       .option("header", "true") \
       .option("inferSchema","true")\
       .load(url) 

data.cache() #for faster re-use

DataFrame[Id: int, SepalLengthCm: double, SepalWidthCm: double, PetalLengthCm: double, PetalWidthCm: double, Species: string]

<div style="
    border-left: 5px solid #2E7D32;
    padding: 14px 18px;
    margin: 20px 0;
    border-radius: 6px;
">
<p style="
    font-size: 1.30em;
    font-weight: 700;
    margin: 0 0 8px 0;
">
<strong>Nota del autor</strong>

<p>
Spark permite cargar el archivo CSV directamente como un DataFrame distribuido. Las opciones header e inferSchema permiten interpretar correctamente los nombres y tipos de las columnas. Además, <code>cache()</code> indica que el DataFrame será reutilizado posteriormente, evitando repetir el calculo del dataframe y haciendo mas eficiente el procesamiento.
</p>

</div>

## Data Exploration & Preparation

<div style="
    border-left: 5px solid #2E7D32;
    padding: 14px 18px;
    margin: 20px 0;
    border-radius: 6px;
">
<p style="
    font-size: 1.30em;
    font-weight: 700;
    margin: 0 0 8px 0;
">
<strong>Nota del autor</strong>

<p>
El DataFrame es la estructura principal utilizada para representar los datos en Spark. Antes de entrenar un modelo, la exploración permite conocer su tamaño, esquema, distribución y estadísticas básicas. Esta etapa ayuda a identificar cómo está estructurado el dataset y a tomar decisiones sobre las transformaciones necesarias antes del entrenamiento.
</p>

</div>

In [11]:
#Total records 
data.count()

150

In [12]:
#Data Type
data.printSchema()

root
 |-- Id: integer (nullable = true)
 |-- SepalLengthCm: double (nullable = true)
 |-- SepalWidthCm: double (nullable = true)
 |-- PetalLengthCm: double (nullable = true)
 |-- PetalWidthCm: double (nullable = true)
 |-- Species: string (nullable = true)



In [13]:
#Display records
data.show(5)

+---+-------------+------------+-------------+------------+-----------+
| Id|SepalLengthCm|SepalWidthCm|PetalLengthCm|PetalWidthCm|    Species|
+---+-------------+------------+-------------+------------+-----------+
|  1|          5.1|         3.5|          1.4|         0.2|Iris-setosa|
|  2|          4.9|         3.0|          1.4|         0.2|Iris-setosa|
|  3|          4.7|         3.2|          1.3|         0.2|Iris-setosa|
|  4|          4.6|         3.1|          1.5|         0.2|Iris-setosa|
|  5|          5.0|         3.6|          1.4|         0.2|Iris-setosa|
+---+-------------+------------+-------------+------------+-----------+
only showing top 5 rows


In [14]:
#Records per Species
data.groupBy('species').count().show()

+---------------+-----+
|        species|count|
+---------------+-----+
| Iris-virginica|   50|
|    Iris-setosa|   50|
|Iris-versicolor|   50|
+---------------+-----+



In [15]:
#Dataset Summary Stats
data.describe().show()

26/09/20 15:51:38 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


+-------+------------------+------------------+-------------------+------------------+------------------+--------------+
|summary|                Id|     SepalLengthCm|       SepalWidthCm|     PetalLengthCm|      PetalWidthCm|       Species|
+-------+------------------+------------------+-------------------+------------------+------------------+--------------+
|  count|               150|               150|                150|               150|               150|           150|
|   mean|              75.5| 5.843333333333335| 3.0540000000000007|3.7586666666666693|1.1986666666666672|          NULL|
| stddev|43.445367992456916|0.8280661279778637|0.43359431136217375| 1.764420419952262|0.7631607417008414|          NULL|
|    min|                 1|               4.3|                2.0|               1.0|               0.1|   Iris-setosa|
|    max|               150|               7.9|                4.4|               6.9|               2.5|Iris-virginica|
+-------+------------------+----

Inorder for our model to make predictions the Species aka Label column should be a numerical value (models don't like string!). To achieve this we shall use String Indexing on the Species columns

In [16]:
#String Indexing the Species column
SIndexer = StringIndexer(inputCol='species', outputCol='species_indx')
data = SIndexer.fit(data).transform(data)

#Inspect the dataset
data.show(5)


+---+-------------+------------+-------------+------------+-----------+------------+
| Id|SepalLengthCm|SepalWidthCm|PetalLengthCm|PetalWidthCm|    Species|species_indx|
+---+-------------+------------+-------------+------------+-----------+------------+
|  1|          5.1|         3.5|          1.4|         0.2|Iris-setosa|         0.0|
|  2|          4.9|         3.0|          1.4|         0.2|Iris-setosa|         0.0|
|  3|          4.7|         3.2|          1.3|         0.2|Iris-setosa|         0.0|
|  4|          4.6|         3.1|          1.5|         0.2|Iris-setosa|         0.0|
|  5|          5.0|         3.6|          1.4|         0.2|Iris-setosa|         0.0|
+---+-------------+------------+-------------+------------+-----------+------------+
only showing top 5 rows


<div style="
    border-left: 5px solid #2E7D32;
    padding: 14px 18px;
    margin: 20px 0;
    border-radius: 6px;
">
<p style="
    font-size: 1.30em;
    font-weight: 700;
    margin: 0 0 8px 0;
">
<strong>Nota del autor</strong>

<p>
<code>StringIndexer</code> transforma las categorías de una columna en índices numéricos. Esto permite representar el label en un formato que los algoritmos de aprendizaje de maquina puedan entender.
</p>

</div>

## Feature Engineering

The Spark model needs two columns: “label” and “features” and we are not going to do much feature engineering because we want to focus on the mechanics of training the model in Spark. 

So, creating a seperate dataframe with re-ordered columns, then defining an input data using Dense Vector. A Dense Vector is a local vector that is backed by a double array that represents its entry values. In other words, it's used to store arrays of values for use in PySpark.


In [17]:
#creating a seperate dataframe with re-ordered columns
df = data.select("species_indx","SepalLengthCm", "SepalWidthCm", "PetalLengthCm", "PetalWidthCm")

#Inspect the dataframe
df.show(5)

+------------+-------------+------------+-------------+------------+
|species_indx|SepalLengthCm|SepalWidthCm|PetalLengthCm|PetalWidthCm|
+------------+-------------+------------+-------------+------------+
|         0.0|          5.1|         3.5|          1.4|         0.2|
|         0.0|          4.9|         3.0|          1.4|         0.2|
|         0.0|          4.7|         3.2|          1.3|         0.2|
|         0.0|          4.6|         3.1|          1.5|         0.2|
|         0.0|          5.0|         3.6|          1.4|         0.2|
+------------+-------------+------------+-------------+------------+
only showing top 5 rows


**Note:** Observe that the species column which is our label (aka Target) is now at beginning of the dataframe

In [18]:
# Define the `input_data` as Dense Vector
input_data = df.rdd.map(lambda x: (x[0], DenseVector(x[1:])))

<div style="
    border-left: 5px solid #2E7D32;
    padding: 14px 18px;
    margin: 20px 0;
    border-radius: 6px;
">
<p style="
    font-size: 1.30em;
    font-weight: 700;
    margin: 0 0 8px 0;
">
<strong>Nota del autor</strong>

<p>
<code>DenseVector</code> agrupa las características de una observación en un único vector numérico. En lugar de manejar cada feature como una columna independiente, Spark puede procesarlas como una sola entrada.
</p>

</div>

**Note:** Observe the definition of the Dense Vector. So,when we create a new indexed dataframe(below) the machine understands that the first column is a Label (Target) and the remaining columns are Features.

In [19]:
# Creating a new Indexed Dataframe
df_indx = spark.createDataFrame(input_data, ["label", "features"])

In [20]:
#view the indexed dataframe
df_indx.show(5)

+-----+-----------------+
|label|         features|
+-----+-----------------+
|  0.0|[5.1,3.5,1.4,0.2]|
|  0.0|[4.9,3.0,1.4,0.2]|
|  0.0|[4.7,3.2,1.3,0.2]|
|  0.0|[4.6,3.1,1.5,0.2]|
|  0.0|[5.0,3.6,1.4,0.2]|
+-----+-----------------+
only showing top 5 rows


<div style="
    border-left: 5px solid #2E7D32;
    padding: 14px 18px;
    margin: 20px 0;
    border-radius: 6px;
">
<p style="
    font-size: 1.30em;
    font-weight: 700;
    margin: 0 0 8px 0;
">
<strong>Nota del autor</strong>

<p>
Spark ML representa la entrada de un modelo mediante una columna features y el resultado esperado mediante label.
</p>

</div>

## Data Scaling

This is also known as Feature Scaling. It is a method of normalizing the features of the data. Scaling can make a difference between a weak machine learning model and a better one. 

In this tutorial we will use a Standard Scaler to scale our feature data. Apache Spark has a Standard Scaler library to do the job.

<div style="
    border-left: 5px solid #2E7D32;
    padding: 14px 18px;
    margin: 20px 0;
    border-radius: 6px;
">
<p style="
    font-size: 1.30em;
    font-weight: 700;
    margin: 0 0 8px 0;
">
<strong>Nota del autor</strong>

<p>
El escalado busca llevar las diferentes características (features) a una escala comparable. Esto puede ser importante para algoritmos sensibles a la magnitud de las variables.
</p>

</div>

In [21]:
#Initialize Standard Scaler
stdScaler = StandardScaler(inputCol="features", outputCol="features_scaled")

#Fit the Standard Scaler to the indexed Dataframe
scaler = stdScaler.fit(df_indx)

#Transform the dataframe
df_scaled =scaler.transform(df_indx)

In [22]:
#Viewing the Scaled Data
df_scaled.show(5)

+-----+-----------------+--------------------+
|label|         features|     features_scaled|
+-----+-----------------+--------------------+
|  0.0|[5.1,3.5,1.4,0.2]|[6.15892840883878...|
|  0.0|[4.9,3.0,1.4,0.2]|[5.9174018045706,...|
|  0.0|[4.7,3.2,1.3,0.2]|[5.67587520030241...|
|  0.0|[4.6,3.1,1.5,0.2]|[5.55511189816831...|
|  0.0|[5.0,3.6,1.4,0.2]|[6.03816510670469...|
+-----+-----------------+--------------------+
only showing top 5 rows


In [23]:
#Dropping the Features column
df_scaled = df_scaled.drop("features")

## Data Split

Just like always, before building a model we shall split our scaled dataset into training & test sets. 
Training Dataset = 90%
Test Dataset = 10%

In [24]:
train_data, test_data = df_scaled.randomSplit([0.9, 0.1], seed = 12345)

In [25]:
#Inspect Training Data
train_data.show(5)

+-----+--------------------+
|label|     features_scaled|
+-----+--------------------+
|  0.0|[5.19282199176603...|
|  0.0|[5.31358529390013...|
|  0.0|[5.31358529390013...|
|  0.0|[5.31358529390013...|
|  0.0|[5.43434859603422...|
+-----+--------------------+
only showing top 5 rows


<div style="
    border-left: 5px solid #2E7D32;
    padding: 14px 18px;
    margin: 20px 0;
    border-radius: 6px;
">
<p style="
    font-size: 1.30em;
    font-weight: 700;
    margin: 0 0 8px 0;
">
<strong>Nota del autor</strong>

<p>
El conjunto de entrenamiento se utiliza para que el modelo aprenda de los parametros al identificar patrones, encontrar relacion para asi minimizar su taza de errro. El conjunto de prueba permite evaluar el comportamiento del modelo con datos que este no ha visto anteriormente. En este caso el dataset se particiona en un 90% entrenamiento, 10% pruebas.
</p>

</div>

## Build, Train & Evaluate Model

In this step we will create multiple models, train them on our scaled dataset and then compare their accuracy.

In [26]:
model = ['Decision Tree','Random Forest','Naive Bayes']
model_results = []

<div style="
    border-left: 5px solid #2E7D32;
    padding: 14px 18px;
    margin: 20px 0;
    border-radius: 6px;
">
<p style="
    font-size: 1.30em;
    font-weight: 700;
    margin: 0 0 8px 0;
">
<strong>Nota del autor</strong>

<p>
Se construyeron los siguientes tres modelos:
    <ol>
    <li> <strong>Decision Tree</strong> toma decisiones mediante una estructura de reglas que divide progresivamente los datos según sus características. </li>
    <li><strong>Random Forest</strong> combina múltiples decision trees para obtener una predicción conjunta y reducir la dependencia de un único árbol.</li> 
     <li><strong>Naive Bayes</strong> utiliza un enfoque probabilístico para estimar la clase más probable a partir de las características, asumiendo independencia entre ellas. </li>   
</ol>
</p>

</div>

In [27]:
# -- Decision Tree Classifier --

dtc = DecisionTreeClassifier(labelCol="label", featuresCol="features_scaled")          #instantiate the model
dtc_model = dtc.fit(train_data)                                                        #train the model
dtc_pred = dtc_model.transform(test_data)                                              #model predictions

#Evaluate the Model
evaluator = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction", metricName="accuracy")
dtc_acc = evaluator.evaluate(dtc_pred)
#print("Decision Tree Classifier Accuracy =", '{:.2%}'.format(dtc_acc))
model_results.extend([[model[0],'{:.2%}'.format(dtc_acc)]])                               #appending to list
    

In [28]:
# -- Random Forest Classifier --

rfc = RandomForestClassifier(labelCol="label", featuresCol="features_scaled", numTrees=10)          #instantiate the model
rfc_model = rfc.fit(train_data)                                                                     #train the model
rfc_pred = rfc_model.transform(test_data)                                                           #model predictions

#Evaluate the Model
evaluator = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction", metricName="accuracy")
rfc_acc = evaluator.evaluate(rfc_pred)
#print("Random Forest Classifier Accuracy =", '{:.2%}'.format(rfc_acc))
model_results.extend([[model[1],'{:.2%}'.format(rfc_acc)]])                                            #appending to list

In [29]:
# -- Naive Bayes Classifier --

nbc = NaiveBayes(smoothing=1.0,modelType="multinomial", labelCol="label",featuresCol="features_scaled")    #instantiate the model
nbc_model = nbc.fit(train_data)                                                                          #train the model
nbc_pred = nbc_model.transform(test_data)                                                                #model predictions

#Evaluate the Model
evaluator = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction", metricName="accuracy")
nbc_acc = evaluator.evaluate(nbc_pred)
#print("Naive Bayes Accuracy =", '{:.2%}'.format(nbc_acc))
model_results.extend([[model[2],'{:.2%}'.format(nbc_acc)]])                                            #appending to list

26/09/20 15:52:35 WARN InstanceBuilder: Failed to load implementation from:dev.ludovic.netlib.blas.JNIBLAS


In [30]:
#freeing memory
gc.collect()

277

<div style="
    border-left: 5px solid #2E7D32;
    padding: 14px 18px;
    margin: 20px 0;
    border-radius: 6px;
">
<p style="
    font-size: 1.30em;
    font-weight: 700;
    margin: 0 0 8px 0;
">
<strong>Nota del autor</strong>

<p>
El flujo de un modelo de clasificación en Spark sigue tres pasos principales
    <ol>
  <li> <code>fit()</code> aprende a partir de los datos de entrenamiento</li>
  <li> <code>transform()</code> genera predicciones</li>
  <li>El evaluador mide qué tan bien coinciden con las etiquetas reales.</li>
</ol>

</p>

</div>

Tabulating the results.

In [31]:
print (tabulate(model_results, headers=["Classifier Models", "Accuracy"]))

Classifier Models    Accuracy
-------------------  ----------
Decision Tree        90.91%
Random Forest        100.00%
Naive Bayes          100.00%


<div style="
    border-left: 5px solid #2E7D32;
    padding: 14px 18px;
    margin: 20px 0;
    border-radius: 6px;
">
<p style="
    font-size: 1.30em;
    font-weight: 700;
    margin: 0 0 8px 0;
">
<strong>Nota del autor</strong>

<p>
Una accuracy del 100% no significa que el modelo prediga perfectamente. En este caso, el conjunto de Iris es pequeño y se utilizó una única partición 90/10, por lo que los resultados podrian variar con otra particion de datos.
</p>

</div>

<div style="padding: 10px 15px; border-left: 3px solid #555;">

<h3>Conclusiones</h3>
<ul>
<li>Spark permite estructurar un flujo completo de machine learning sobre DataFrames, desde la exploración y transformación de los datos hasta el entrenamiento y evaluación de modelos.
</li>
<li>El preprocesamiento es una parte fundamental del entrenamiento. La transformación de la variable objetivo mediante StringIndexer, la construcción del vector de características y el escalado permiten adaptar los datos al formato requerido por Spark ML.</li>

</ul>
</div>